# Using the BaseDispatchingComposite Module in baseobjects.composition

## Introduction

`BaseDispatchingComposite` extends `BaseComposite` by adding a mechanism to dynamically determine which component types to instantiate based on input arguments. It uses a `NamespaceClassRegistry` to store and retrieve component classes, allowing for flexible component creation at runtime.

This pattern is useful when you have many potential component types and you want to decide which ones to use based on configuration, data, or user input, without hardcoding every possibility into the composite class.

This tutorial covers:
- Setting up a component registry
- Registering component classes
- Implementing the `dispatch_component_types` method
- Creating composites with dynamically dispatched components

**Prerequisites:**
- Basic familiarity with `BaseComposite` and `BaseComponent`
- Installed package: `baseobjects`

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

We'll need `BaseDispatchingComposite`, `BaseComponent`, and `NamespaceClassRegistry`.


In [ ]:
from typing import Any, ClassVar
from baseobjects.composition import BaseDispatchingComposite, BaseComponent
from baseobjects.classregistration import NamespaceClassRegistry


## Core Functionality

The core feature of `BaseDispatchingComposite` is its ability to map initialization arguments to specific component classes using a registry.

### Key Concepts

1. **NamespaceClassRegistry**: A container for mapping namespaces and names to classes.
2. **dispatch_component_types**: A method that the user must implement to define how components are selected from the registry.

### Defining Components

Let's define a few simple text processing components.


In [ ]:
class TextComponent(BaseComponent):
    """Base for text processing components."""
    def process(self, text: str) -> str:
        return text

class UppercaseComponent(TextComponent):
    def process(self, text: str) -> str:
        return text.upper()

class LowercaseComponent(TextComponent):
    def process(self, text: str) -> str:
        return text.lower()

class PrefixComponent(TextComponent):
    def __init__(self, *args, prefix="> ", **kwargs):
        super().__init__(*args, **kwargs)
        self.prefix = prefix

    def process(self, text: str) -> str:
        return f"{self.prefix}{text}"


### Setting up the Registry

`BaseDispatchingComposite` uses a registry to map identifiers to component classes.


In [ ]:
# Create a registry
text_registry = NamespaceClassRegistry()

# Register the components
text_registry.register_class(UppercaseComponent, namespace="format", name="upper")
text_registry.register_class(LowercaseComponent, namespace="format", name="lower")
text_registry.register_class(PrefixComponent, namespace="extra", name="prefix")


### Creating the Dispatching Composite

To use dispatching, you need to:
1. Assign the `component_types_registry`.
2. Implement `dispatch_component_types`.


In [ ]:
class TextProcessor(BaseDispatchingComposite):
    """A composite that dispatches components from a registry."""

    # Assign the registry we created
    component_types_registry = text_registry

    def dispatch_component_types(self, name: str, namespace: str, class_name: str, **kwargs) -> dict[str, tuple[type, dict[str, Any]]]:
        """Dispatches a component from the registry by its name and namespace."""
        # Get the class and its default kwargs from the registry
        component_info = self.component_types_registry.get_class(namespace, class_name, with_kwargs=True, **kwargs)
        # Return a dictionary mapping the instance name to the component info
        return {name: component_info}

    def process(self, text: str) -> str:
        """Process text using all attached components."""
        for component in self.components.values():
            if isinstance(component, TextComponent):
                text = component.process(text)
        return text


### Using the Dispatching Composite

Instantiate the composite and specify which components to create using the identifiers from the registry.


In [ ]:
# Create a processor and dispatch an 'upper' component from 'format' namespace
# We name the instance "formatter"
processor = TextProcessor(name="formatter", namespace="format", class_name="upper")

print(f"Components: {list(processor.components.keys())}")
print(f"Result: {processor.process('hello')}")


## Module Interaction

`BaseDispatchingComposite` interacts heavily with `NamespaceClassRegistry`. It uses the registry to lookup classes and their default keyword arguments, facilitating a decoupled architecture where the composite doesn't need to know the specific component classes beforehand.


## Advanced Features

### Adding More Components via Dispatch

The `construct` method can be used to add more components later using the same dispatching mechanism.


In [ ]:
# Add a prefix component
processor.construct(name="prefiller", namespace="extra", class_name="prefix", prefix="RE: ")

print(f"Components: {list(processor.components.keys())}")
print(f"Result: {processor.process('hello')}")


### Overriding at Runtime

The dispatching mechanism still allows passing specific `component_types` to bypass the registry for a particular component.


In [ ]:
class ReverseComponent(TextComponent):
    def process(self, text: str) -> str:
        return text[::-1]

# Create a processor with one dispatched component and one explicitly provided component
custom_processor = TextProcessor(
    name="lower", namespace="format", class_name="lower",
    component_types={"reverser": (ReverseComponent, {})}
)

print(f"Components: {list(custom_processor.components.keys())}")
print(f"Result: {custom_processor.process('Hello World')}")


## Examples

Here is an example of using dispatching for a configurable logging system.


In [ ]:
class ConsoleLogger(BaseComponent):
    def log(self, msg): print(f"[CONSOLE] {msg}")

class FileLogger(BaseComponent):
    def log(self, msg): print(f"[FILE] {msg}")

log_registry = NamespaceClassRegistry()
log_registry.register_class(ConsoleLogger, namespace="log", name="console")
log_registry.register_class(FileLogger, namespace="log", name="file")

class MultiLogger(BaseDispatchingComposite):
    component_types_registry = log_registry
    def dispatch_component_types(self, log_type="console", **kwargs):
        return {"logger": self.component_types_registry.get_class("log", log_type, with_kwargs=True)}
    def log(self, msg):
        for c in self.components.values(): c.log(msg)

logger = MultiLogger(log_type="file")
logger.log("Hello")


## API Highlights

- **`BaseDispatchingComposite`**: Extends `BaseComposite` with dispatching.
  - `component_types_registry`: Class attribute holding the `NamespaceClassRegistry`.
  - `dispatch_component_types(*args, **kwargs)`: Abstract method to select components.
  - `construct(*args, **kwargs)`: Dynamically adds components using dispatching.

## Troubleshooting / FAQs

- **Problem**: `KeyError` when dispatching.
  - **Solution**: Ensure the `namespace` and `class_name` (or whatever identifiers you use) are correctly registered in the `component_types_registry`.

- **Problem**: Components are not being created.
  - **Solution**: Verify that `dispatch_component_types` returns a dictionary in the format `{name: (class, kwargs)}`.

## Conclusion and Next Steps

`BaseDispatchingComposite` provides a powerful way to decouple your composite objects from the specific component implementations they use. By using a registry and a dispatching method, you can build highly configurable and extensible systems.

- **Next**: Explore `DispatchableComposite` for combined class and component dispatching.
- **Reference**: See `src/baseobjects/composition/basedispatchingcomposite.py` for implementation details.
